In [ ]:
# 0: INSTALL REQUIRED LIBRARIES
# FUNCTION:
#   Installs the mlxtend library which provides efficient implementations of frequent pattern mining algorithms such
# as Apriori and FP-Growth.
# MECHANISM:
#   The pip installer downloads and installs mlxtend into the current Python environment so that association rule mining
# functions can be imported and used later in the notebook.
# EXPECTED OUTPUT:
#   A confirmation that mlxtend has been installed successfully without any errors.
# CONSIDERATIONS:
#   If the runtime is restarted, this cell should be executed again to ensure the library remains available.

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

!pip install -q mlxtend

In [ ]:
# 1: IMPORT LIBRARIES + BASIC SETTINGS
# FUNCTION:
#   Loads all required Python libraries used for data processing, visualization, performance benchmarking, and association rule mining.
# MECHANISM:
#   - pandas and numpy handle structured data operations
#   - matplotlib is used for visualizing patterns and results
#   - perf_counter measures algorithm runtime
#   - tracemalloc tracks memory usage during algorithm execution
#   - mlxtend provides Apriori, FP-Growth, and association rule generation functions.
# EXPECTED OUTPUT:
#   The libraries load successfully without producing errors.
# CONSIDERATIONS:
#   If an import error occurs, ensure the required package has been installed using pip before running this block.

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from time import perf_counter
import tracemalloc

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# Improve dataframe display formatting
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# DISPLAY SETTINGS FOR CLEAN OUTPUTS
# FUNCTION:
#   Makes pandas tables easier to read in the notebook.
# MECHANISM:
#   Limits number of rows/columns displayed and rounds decimals.
# EXPECTED OUTPUT:
#   Tables appear smaller and cleaner.
# CONSIDERATIONS:
#   Only affects how tables display, not the data itself.


import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 10)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
# 2: LOAD DATASET
# FUNCTION:
#   Loads the accidental drug-related deaths dataset
# MECHANISM:
#   The pandas read_csv() function reads the CSV file and stores the dataset as a DataFrame, allowing structured manipulation
# of rows and columns.
# EXPECTED OUTPUT:
#   - Confirmation message that the dataset is loaded
#   - Display of dataset dimensions (rows, columns)
#   - Preview of the first few rows
# CONSIDERATIONS:
#   Ensure the dataset file is located in the working directory or uploaded into the notebook environment before running.

import pandas as pd

df = pd.read_excel("/content/A2_Cleaned Dataset.xlsx")

print("Dataset successfully loaded.")
print("Dataset shape:", df.shape)

display(df.head(3))

Dataset successfully loaded.
Dataset shape: (11981, 48)


,date,date_type,age,sex,race,...,any_opioid,other,residencecitygeo,injurycitygeo,deathcitygeo
0,05/29/2012,Date of death,37,Male,Black,...,NaN,Unknown,"STAMFORD, CT\n(41.051924, -73.539475)","STAMFORD, CT\n(41.051924, -73.539475)","CT\n(41.575155, -72.738288)"
1,06/27/2012,Date of death,37,Male,White,...,NaN,Unknown,"NORWICH, CT\n(41.524304, -72.075821)","NORWICH, CT\n(41.524304, -72.075821)","Norwich, CT\n(41.524304, -72.075821)"
2,03/24/2014,Date of death,28,Male,White,...,NaN,Unknown,"HEBRON, CT\n(41.658069, -72.366324)","HEBRON, CT\n(41.658069, -72.366324)","Marlborough, CT\n(41.632043, -72.461309)"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3: SELECT DRUG COLUMNS
# FUNCTION:
#   Extract the drug indicator columns from the cleaned dataset.
# MECHANISM:
#   These columns represent substances detected in overdose
# cases and will form the basis of the transaction matrix.
# EXPECTED OUTPUT:
#   Dataframe containing only drug-related attributes.
# CONSIDERATIONS:
#   Only columns relevant to drug presence should be selected.

drug_columns = df.columns

df_drugs = df[drug_columns].copy()

print("Number of drug variables:", len(drug_columns))
df_drugs.head()

Number of drug variables: 48


,date,date_type,age,sex,race,...,any_opioid,other,residencecitygeo,injurycitygeo,deathcitygeo
0,05/29/2012,Date of death,37,Male,Black,...,NaN,Unknown,"STAMFORD, CT\n(41.051924, -73.539475)","STAMFORD, CT\n(41.051924, -73.539475)","CT\n(41.575155, -72.738288)"
1,06/27/2012,Date of death,37,Male,White,...,NaN,Unknown,"NORWICH, CT\n(41.524304, -72.075821)","NORWICH, CT\n(41.524304, -72.075821)","Norwich, CT\n(41.524304, -72.075821)"
2,03/24/2014,Date of death,28,Male,White,...,NaN,Unknown,"HEBRON, CT\n(41.658069, -72.366324)","HEBRON, CT\n(41.658069, -72.366324)","Marlborough, CT\n(41.632043, -72.461309)"
3,12/31/2014,Date of death,26,Female,White,...,NaN,Unknown,"BALTIC, CT\n(41.617221, -72.085031)","CT\n(41.575155, -72.738288)","Baltic, CT\n(41.617221, -72.085031)"
4,01/16/2016,Date of death,41,Male,White,...,Y,Unknown,"SHELTON, CT\n(41.316843, -73.092968)","SHELTON, CT\n(41.316843, -73.092968)","Bridgeport, CT\n(41.179195, -73.189476)"


In [ ]:
# 3: SELECT DRUG INDICATOR COLUMNS
# FUNCTION:
#   Identifies and extracts the drug-related indicator columns that represent substances involved in each death case.
# MECHANISM:
#   A predefined list of drug variables is checked against the dataset columns to ensure only existing columns are used.
# This prevents errors if column names vary slightly.
# EXPECTED OUTPUT:
#   - Number of drug columns detected
#   - Any missing drug columns from the expected list
# CONSIDERATIONS:
#   Column names must match those used in previous work to maintain consistency in previous preprocessing steps.

# Convert column names to lowercase for easier matching
df.columns = df.columns.str.lower()

# Keywords that indicate drug columns
drug_keywords = [
    "heroin","cocaine","fentanyl","opioid","benzodiazepine",
    "methadone","ethanol","amphetamine","morphine",
    "oxycodone","hydrocodone","hydromorphone",
    "xylazine","gabapentin","tramad"
]

# Detect columns containing those keywords
drug_columns = [
    col for col in df.columns
    if any(keyword in col for keyword in drug_keywords)
]

print("Drug columns detected:", len(drug_columns))
print(drug_columns)

# Create drug dataframe
df_drugs = df[drug_columns].copy()

print("\nPreview of drug dataset:")
print(df_drugs.iloc[:5, :8].to_string())

Drug columns detected: 19
['heroin', 'heroin_death_certificate_dc', 'cocaine', 'fentanyl', 'fentanyl_analogue', 'oxycodone', 'ethanol', 'hydrocodone', 'benzodiazepine', 'methadone', 'meth_amphetamine', 'tramad', 'hydromorphone', 'morphine_not_heroin', 'xylazine', 'gabapentin', 'heroin_morph_codeine', 'other_opioid', 'any_opioid']

Preview of drug dataset:
  heroin heroin_death_certificate_dc cocaine fentanyl fentanyl_analogue oxycodone ethanol hydrocodone
0    NaN                         NaN       Y      NaN               NaN       NaN     NaN         NaN
1      Y                         NaN     NaN      NaN               NaN       NaN     NaN         NaN
2      Y                         NaN     NaN      NaN               NaN       NaN     NaN         NaN
3      Y                         NaN     NaN      NaN               NaN       NaN     NaN         NaN
4    NaN                         NaN     NaN        Y               NaN       NaN     NaN         NaN


In [ ]:
# 4: CREATE BINARY DRUG MATRIX
# FUNCTION:
#   Converts drug indicator columns into binary values where 1 indicates the presence of a substance
# and 0 indicates its absence in a case.
# MECHANISM:
#   - A subset DataFrame containing only drug columns is created
#   - Each column is converted using .notna().astype(int)
#   - This produces a binary matrix suitable for association
#   rule mining algorithms.
# EXPECTED OUTPUT:
#   A table showing drug indicators represented as binary values (0 or 1).
# CONSIDERATIONS:
#   Binary encoding is required for Apriori and FP-Growth to correctly identify frequent itemsets.

drug_cols = [
    "any_opioid",
    "benzodiazepine",
    "cocaine",
    "ethanol",
    "fentanyl",
    "fentanyl_analogue",
    "heroin",
    "heroin_death_certificate_dc",
    "heroin_morph_codeine",
    "hydrocodone",
    "meth_amphetamine",
    "methadone",
    "opiate_nos",
    "other_opioid",
    "oxycodone"
]

# Keep only columns that exist
drug_cols_existing = [c for c in drug_cols if c in df.columns]

print("Drug columns detected:", len(drug_cols_existing))
print("Missing drug columns:", sorted(list(set(drug_cols) - set(drug_cols_existing))))

# Creating drug-only dataframe
df_drugs = df[drug_cols_existing].copy()

# Converting to binary: 1 = present (not null), 0 = absent (null)
for c in df_drugs.columns:
    df_drugs[c] = df_drugs[c].notna().astype(int)

print("Binary drug matrix created (0/1).")

# Print a clean preview
print(df_drugs.iloc[:5, :8].to_string(index=True))

Drug columns detected: 15
Missing drug columns: []
Binary drug matrix created (0/1).
   any_opioid  benzodiazepine  cocaine  ethanol  fentanyl  fentanyl_analogue  heroin  heroin_death_certificate_dc
0           0               0        1        0         0                  0       0                            0
1           0               0        0        0         0                  0       1                            0
2           0               0        0        0         0                  0       1                            0
3           0               0        0        0         0                  0       1                            0
4           1               0        0        0         1                  0       0                            0


In [ ]:
# 5: CREATE TRANSACTION DATASET
# FUNCTION:
#   Converts each case into a transaction containing the list of drugs present in that case.
# MECHANISM:
#   For each row, column names with value 1 are extracted and
# stored as a list representing the substances involved in that particular case.
# EXPECTED OUTPUT:
#   - Total number of cases
#   - Number of valid transactions
#   - Example transaction showing detected substances
# CONSIDERATIONS:
#   Empty transactions (cases with no detected substances) are
# removed because they do not contribute to association rule mining.

transactions = []

for _, row in df_drugs.iterrows():
    present_items = row.index[row == 1].tolist()
    transactions.append(present_items)

transactions_nonempty = [t for t in transactions if len(t) > 0]

print("Total cases:", len(transactions))
print("Valid transactions:", len(transactions_nonempty))
print("Example transaction:", transactions_nonempty[0])

Total cases: 11981
Valid transactions: 11845
Example transaction: ['cocaine']


In [ ]:
# 6: IMPORT ALGORITHMS
# FUNCTION:
#   Imports Apriori, FP-Growth, and rule generator.
# EXPECTED OUTPUT:
#   A confirmation message.

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

print("Algorithms ready")

Algorithms ready


In [ ]:
# 7: Data Preparation
# FUNCTION:
#   Uses the binary drug matrix for association analysis.
# MECHANISM:
#   The previously created binary dataframe (df_drugs) is copied into a new variable called transaction_matrix.
# Each row represents a case and each column represents a drug.
# EXPECTED OUTPUT:
#   Dataset shape and first rows.
# CONSIDERATIONS:
#   The dataset must contain only binary values (0 or 1) because Apriori and FP-Growth require binary transaction data.

transaction_matrix = df_drugs.copy()

print("Dataset size:", transaction_matrix.shape)

transaction_matrix.head()

Dataset size: (11981, 15)


,any_opioid,benzodiazepine,cocaine,ethanol,fentanyl,...,meth_amphetamine,methadone,opiate_nos,other_opioid,oxycodone
0,0,0,1,0,0,...,0,0,0,0,0
1,0,0,0,0,0,...,0,0,0,0,0
2,0,0,0,0,0,...,0,0,0,0,0
3,0,0,0,0,0,...,0,0,0,0,0
4,1,0,0,0,1,...,0,0,0,0,0


In [ ]:
# 8: APPLY APRIORI ALGORITHM
# FUNCTION:
#   Identifies frequent drug combinations using the Apriori algorithm.
# MECHANISM:
#   Apriori scans the transaction matrix to find combinations of drugs that appear together frequently.
# It uses a minimum support threshold to filter out rare combinations.
# EXPECTED OUTPUT:
#   A table containing frequent drug combinations and their support values.
# CONSIDERATIONS:
#   A lower support threshold will generate more combinations but will also increase computation time.

import time

start = time.time()

frequent_itemsets_apriori = apriori(
    transaction_matrix,
    min_support=0.05,
    use_colnames=True
)

end = time.time()

apriori_runtime = end - start

print("Apriori runtime:", apriori_runtime)

frequent_itemsets_apriori.head()

Apriori runtime: 0.08283638954162598


,support,itemsets
0,0.747,(any_opioid)
1,0.227,(benzodiazepine)
2,0.382,(cocaine)
3,0.267,(ethanol)
4,0.672,(fentanyl)


In [ ]:
# 9: GENERATE ASSOCIATION RULES (APRIORI)
# FUNCTION:
#   Generates association rules from the frequent itemsets identified by the Apriori algorithm.
# MECHANISM:
#   The association_rules function evaluates relationships between drug combinations using metrics such as confidence and lift.
# EXPECTED OUTPUT:
#   A dataframe showing rules such as Drug A → Drug B along with support, confidence, and lift values.
# CONSIDERATIONS:
#   Higher lift values indicate stronger relationships between drugs appearing together in overdose cases.

rules_apriori = association_rules(
    frequent_itemsets_apriori,
    metric="confidence",
    min_threshold=0.6
)

rules_apriori.head()

,antecedents,consequents,antecedent support,consequent support,support,...,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(benzodiazepine),(any_opioid),0.227,0.747,0.165,...,0.931,-0.032,0.204,-0.074,0.475
1,(cocaine),(any_opioid),0.382,0.747,0.292,...,1.076,0.038,0.349,0.071,0.578
2,(ethanol),(any_opioid),0.267,0.747,0.203,...,1.053,0.023,0.250,0.050,0.516
3,(any_opioid),(fentanyl),0.747,0.672,0.604,...,1.720,0.670,0.742,0.419,0.854
4,(fentanyl),(any_opioid),0.672,0.747,0.604,...,2.520,0.517,0.742,0.603,0.854


In [ ]:
# 10: APPLY FP-GROWTH ALGORITHM
# FUNCTION:
# Identifies frequent drug combinations using the FP-Growth algorithm.
# MECHANISM:
#   FP-Growth builds a compressed tree structure called an FP-tree to efficiently discover frequent itemsets without
# generating candidate combinations like Apriori.
# EXPECTED OUTPUT:
#   A table showing frequent drug combinations and their support values.
# CONSIDERATIONS:
#   FP-Growth is generally faster and more efficient than Apriori when working with large datasets.

start = time.time()

frequent_itemsets_fp = fpgrowth(
    transaction_matrix,
    min_support=0.05,
    use_colnames=True
)

end = time.time()

fp_runtime = end - start

print("FP-Growth runtime:", fp_runtime)

frequent_itemsets_fp.head()

FP-Growth runtime: 4.343732833862305


,support,itemsets
0,0.382,(cocaine)
1,0.299,(heroin)
2,0.747,(any_opioid)
3,0.672,(fentanyl)
4,0.085,(oxycodone)


In [ ]:
# 11: GENERATE ASSOCIATION RULES (FP-GROWTH)
# FUNCTION:
#   Generates association rules from the frequent itemsets discovered using the FP-Growth algorithm.
# MECHANISM:
#   The association_rules function evaluates relationships between itemsets using metrics such as confidence and lift.
# EXPECTED OUTPUT:
#   A table of association rules representing relationships between drugs appearing together in overdose cases.
# CONSIDERATIONS:
#   Rules with high confidence and lift indicate strong drug co-occurrence patterns.

rules_fp = association_rules(
    frequent_itemsets_fp,
    metric="confidence",
    min_threshold=0.6
)

rules_fp.head()

,antecedents,consequents,antecedent support,consequent support,support,...,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(cocaine),(any_opioid),0.382,0.747,0.292,...,1.076,0.038,0.349,0.071,0.578
1,(cocaine),(fentanyl),0.382,0.672,0.277,...,1.196,0.120,0.357,0.164,0.569
2,"(any_opioid, cocaine)",(fentanyl),0.292,0.672,0.256,...,2.660,0.330,0.362,0.624,0.629
3,"(cocaine, fentanyl)",(any_opioid),0.277,0.747,0.256,...,3.325,0.265,0.334,0.699,0.633
4,(cocaine),"(any_opioid, fentanyl)",0.382,0.604,0.256,...,1.201,0.160,0.351,0.167,0.547


In [ ]:
# 12: COMPARE ALGORITHM PERFORMANCE
# FUNCTION:
#   Compares the execution time of the Apriori and FP-Growth algorithms.
# MECHANISM:
#   The runtime values measured earlier are printed and compared to determine which algorithm executed faster.
# EXPECTED OUTPUT:
#   A printed comparison of the runtime of Apriori and FP-Growth algorithms.
# CONSIDERATIONS:
#   FP-Growth is typically faster because it avoids generating candidate itemsets using an FP-tree structure.

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Apriori runtime:", apriori_runtime)
print("FP-Growth runtime:", fp_runtime)

if fp_runtime < apriori_runtime:
    print("FP-Growth algorithm is faster.")
else:
    print("Apriori algorithm is faster.")

Apriori runtime: 0.08283638954162598
FP-Growth runtime: 4.343732833862305
Apriori algorithm is faster.


In [ ]:
print("Apriori itemsets:", len(frequent_itemsets_apriori))
print("FP-Growth itemsets:", len(frequent_itemsets_fp))

print("Apriori rules:", len(rules_apriori))
print("FP-Growth rules:", len(rules_fp))

Apriori itemsets: 59
FP-Growth itemsets: 59
Apriori rules: 81
FP-Growth rules: 81


In [ ]:
print("FINAL SUMMARY")
print("----------------")

print("Apriori Runtime:", apriori_runtime)
print("FP-Growth Runtime:", fp_runtime)

if fp_runtime < apriori_runtime:
    print("FP-Growth was faster on this dataset.")
else:
    print("Apriori was faster on this dataset.")

print("Both algorithms discovered similar frequent drug combinations.")
print("This confirms consistency of the mining results.")

FINAL SUMMARY
----------------
Apriori Runtime: 0.08283638954162598
FP-Growth Runtime: 4.343732833862305
Apriori was faster on this dataset.
Both algorithms discovered similar frequent drug combinations.
This confirms consistency of the mining results.


In [ ]:
# TOP 10 DRUG ASSOCIATIONS
# FUNCTION:
#   Displays the strongest association rules in a clean table, focusing on the most important metrics for interpretation.
# MECHANISM:
#   - Sorts rules by lift (strongest associations first).
#   - Keeps only the key columns required for reporting.
#   - Limits output to top 10 to avoid messy/crowded results.
# EXPECTED OUTPUT:
#   A neat Top 10 table showing: antecedents, consequents, support, confidence, lift.
# CONSIDERATIONS:
#   Lift is used as a primary "interestingness" measure.
#   If too many weak rules appear, increase min_confidence or min_support.

def top_rules_table(rules_df, top_n=10):
    # Copying to avoid modifying original rules dataframe
    temp = rules_df.copy()

    # Sorting rules by lift then confidence to prioritize stronger patterns
    temp = temp.sort_values(["lift", "confidence"], ascending=False)

    # Keeping only key columns for clean reporting
    temp = temp[["antecedents", "consequents", "support", "confidence", "lift"]]

    # Limiting to top N rows
    return temp.head(top_n)

print("Top 10 Association Rules (Apriori):")
display(top_rules_table(rules_apriori, top_n=10))

print("Top 10 Association Rules (FP-Growth):")
display(top_rules_table(rules_fp, top_n=10))

Top 10 Association Rules (Apriori):


,antecedents,consequents,support,confidence,lift
57,(heroin_death_certificate_dc),"(heroin, heroin_morph_codeine)",0.062,1.000,5.752
77,"(any_opioid, heroin_death_certificate_dc)","(heroin, heroin_morph_codeine)",0.062,1.000,5.752
80,(heroin_death_certificate_dc),"(any_opioid, heroin, heroin_morph_codeine)",0.062,0.997,5.745
47,(heroin_death_certificate_dc),"(any_opioid, heroin_morph_codeine)",0.062,0.997,5.444
78,"(heroin, heroin_death_certificate_dc)","(any_opioid, heroin_morph_codeine)",0.062,0.997,5.444
17,(heroin_death_certificate_dc),(heroin_morph_codeine),0.062,1.000,5.441
45,"(any_opioid, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
55,"(heroin, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
74,"(any_opioid, heroin, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
40,(heroin_death_certificate_dc),"(any_opioid, heroin)",0.062,0.997,5.100


Top 10 Association Rules (FP-Growth):


,antecedents,consequents,support,confidence,lift
47,(heroin_death_certificate_dc),"(heroin, heroin_morph_codeine)",0.062,1.000,5.752
57,"(any_opioid, heroin_death_certificate_dc)","(heroin, heroin_morph_codeine)",0.062,1.000,5.752
60,(heroin_death_certificate_dc),"(any_opioid, heroin, heroin_morph_codeine)",0.062,0.997,5.745
50,(heroin_death_certificate_dc),"(any_opioid, heroin_morph_codeine)",0.062,0.997,5.444
58,"(heroin, heroin_death_certificate_dc)","(any_opioid, heroin_morph_codeine)",0.062,0.997,5.444
42,(heroin_death_certificate_dc),(heroin_morph_codeine),0.062,1.000,5.441
45,"(heroin, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
48,"(any_opioid, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
54,"(any_opioid, heroin, heroin_death_certificate_dc)",(heroin_morph_codeine),0.062,1.000,5.441
53,(heroin_death_certificate_dc),"(any_opioid, heroin)",0.062,0.997,5.100


In [ ]:
# PARAMETER TUNING SUMMARY
# FUNCTION:
#   Creates a simple summary table showing your chosen thresholds.
# MECHANISM:
#   Stores key parameter values (support/confidence) and result sizes (number of itemsets/rules) into a small table.
# EXPECTED OUTPUT:
#   A neat summary table that can be inserted in the report.
# CONSIDERATIONS:
#   Helps justify parameter selection in the write-up.

tuning_summary = pd.DataFrame({
    "Parameter": ["min_support", "min_confidence", "Apriori itemsets", "Apriori rules", "FP itemsets", "FP rules"],
    "Value": [0.05, 0.6, len(frequent_itemsets_apriori), len(rules_apriori), len(frequent_itemsets_fp), len(rules_fp)]
})

display(tuning_summary)

,Parameter,Value
0,min_support,0.050
1,min_confidence,0.600
2,Apriori itemsets,59.000
3,Apriori rules,81.000
4,FP itemsets,59.000
5,FP rules,81.000
